In [1]:
!pip install gradio requests -q

import requests
import pandas as pd
from google.colab import auth
from google.cloud import bigquery

# 1. AUTH & CLIENT (Tek bir kez yapılması yeterli)
auth.authenticate_user()
client = bigquery.Client(project="PROJE_ID_NIZ")

# 2. HISTORICAL DATA (BigQuery Sorgusu)
query = """
SELECT *
FROM `fatihdata.european_soccer_analysis.mart_lig_ve_takim_performans`
LIMIT 1000
"""
df = client.query(query).to_dataframe()

# 3. LIVE API (Canlı Veri Çekme)
API_KEY = "BURAYA_KENDI_API_KEYINIZI_YAZIN"
URL = "https://v3.football.api-sports.io/fixtures"

r = requests.get(
    URL,
    headers={"x-apisports-key": API_KEY},
    params={"live": "all"}
)
live = r.json().get("response", [])

# 4. KONTROL ÇIKTISI
print("Historical (BigQuery):", len(df))
print("Live (API):", len(live))

Historical (BigQuery): 1000
Live (API): 2


In [ ]:
import gradio as gr

# -----------------------------------------------------------------
# GÜVENLİ FİLTRELER VE LİSTELER
# -----------------------------------------------------------------
EUROPE_COUNTRIES = ["England", "France", "Italy", "Spain", "Germany", "Netherlands", "Turkey", "Portugal", "Belgium"]

def is_women(league_name):
    return "Women" in str(league_name)

def advanced_upset_score(home, away, hg, ag, df_historical):
    if ag > hg:
        return "🔥 HIGH RISK (Potential Upset)"
    elif hg == ag:
        return "🟡 MEDIUM RISK (Draw Trend)"
    else:
        return "🟢 LOW RISK (Normal Status)"

def create_match_card(league, country, home, away, hg, ag, risk):
    if "HIGH" in risk:
        bg_color = "#ffebee"; border_color = "#ef5350"; text_color = "#c62828"
    elif "MEDIUM" in risk:
        bg_color = "#fff3e0"; border_color = "#ffb74d"; text_color = "#ef6c00"
    else:
        bg_color = "#f1f8e9"; border_color = "#9ccc65"; text_color = "#33691e"

    return f"""
    <div style="background-color: {bg_color}; border-left: 6px solid {border_color}; padding: 15px; margin-bottom: 15px; border-radius: 6px;">
        <div style="font-size: 12px; color: #666; text-transform: uppercase; font-weight: bold; margin-bottom: 5px;">⚽ {league} ({country})</div>
        <div style="display: flex; justify-content: space-between; align-items: center; margin: 10px 0; gap: 15px;">
            <div style="font-size: 16px; font-weight: 600; width: 35%; color: #000;">🏠 {home}</div>
            <div style="font-size: 22px; font-weight: 800; background: #1e293b; color: #fff; padding: 6px 16px; border-radius: 6px; min-width: 80px; text-align: center;">{hg} - {ag}</div>
            <div style="font-size: 16px; font-weight: 600; width: 35%; text-align: right; color: #000;">🚀 {away}</div>
        </div>
        <div style="font-size: 13px; font-weight: bold; color: {text_color}; margin-top: 5px; padding-top: 5px; border-top: 1px dashed #ccc;">🚨 UPSET STATUS: {risk}</div>
    </div>
    """

# -----------------------------------------------------------------
# GÜVENLİ GRADIO DASHBOARD FUNCTION
# -----------------------------------------------------------------
def dashboard():
    # HAFIZA KONTROLÜ: Eğer 1. Adım hücresi çalışmadıysa hata vermesini engelle
    if 'live' not in globals() or 'df' not in globals():
        return "<p style='text-align:center; color:red; font-size:16px;'>Hata: Önce 1. Adım hü創造 (API ve BigQuery verisini çeken hücre) çalıştırılmalıdır!</p>"

    # API KOTA KONTROLÜ: Eğer gelen veri liste değilse (Sınır dolduysa sözlük döner)
    if not isinstance(live, list):
        return f"<p style='text-align:center; color:red; font-size:16px;'>API Hatası veya Kota Sınırı: {str(live)}</p>"

    html_content = '<div style="max-width: 800px; margin: 0 auto;">'
    match_count = 0

    try:
        for g in live:
            # API yanıtında eksik key gelme ihtimaline karşı güvenli sözlük okuma (.get)
            league_info = g.get("league", {})
            country = league_info.get("country", "")
            league = league_info.get("name", "")

            if country not in EUROPE_COUNTRIES or is_women(league):
                continue

            home = g.get("teams", {}).get("home", {}).get("name", "Unknown Home")
            away = g.get("teams", {}).get("away", {}).get("name", "Unknown Away")
            hg = g.get("goals", {}).get("home") if g.get("goals", {}).get("home") is not None else 0
            ag = g.get("goals", {}).get("away") if g.get("goals", {}).get("away") is not None else 0

            risk = advanced_upset_score(home, away, hg, ag, df)
            html_content += create_match_card(league, country, home, away, hg, ag, risk)
            match_count += 1

        html_content += "</div>"

        if match_count == 0:
            return "<p style='text-align:center; color:#666; font-size:16px; padding:20px;'>Şu an filtrelere uygun canlı Avrupa maçı bulunamadı.</p>"
        return html_content

    except Exception as e:
        # Kod döngü içinde patlarsa gerçek hatayı ekrana basar
        return f"<p style='text-align:center; color:red; font-size:16px;'>Döngü Hatası: {str(e)}</p>"

# -----------------------------------------------------------------
# UI LAUNCH (DEBUG MODU AÇIK)
# -----------------------------------------------------------------
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("<h1 style='text-align: center; color: #2c3e50; margin-top: 20px;'>⚽ EUROPE FOOTBALL DASHBOARD</h1>")
    with gr.Row():
        btn = gr.Button("🔄 REFRESH DATA", variant="primary", size="lg")
    output = gr.HTML(value="<p style='text-align:center; color:#666; padding:20px;'>Verileri yüklemek için yenile butonuna basın.</p>")
    btn.click(fn=dashboard, outputs=output)

app.close()
# theme parametresini buradan kaldırdık, sadece debug ve share bıraktık
app.launch(share=True, debug=True)

/tmp/ipykernel_3715/1590968502.py:86: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b969d82b2a089bf7ce.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
